<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">

<h1><center>RAG with Agent solution</center></h1>

# Installing Dependencies

### requirements.txt

openai==0.28  
ipykernel==6.20.2  
langchain==0.0.352  
wikipedia==1.4.0  
rank_bm25==0.2.2   
tiktoken==0.5.2
faiss-cpu==1.7.4


In [2]:
# !pip install -r requirements.txt

In [3]:
!pip install openai==0.28 ipykernel==6.20.2 langchain==0.0.352 wikipedia==1.4.0 rank_bm25==0.2.2 tiktoken==0.5.2 faiss-cpu==1.7.4

  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.8.0, 1.8.0.post1, 1.9.0, 1.9.0.post1, 1.10.0, 1.11.0, 1.11.0.post1, 1.12.0, 1.13.0, 1.13.1, 1.13.2, 1.14.2, 1.14.3, 1.15.0)
ERROR: No matching distribution found for faiss-cpu==1.7.4


In [4]:
!pip install langchain-openai langchain-community langchain-classic langchain-text-splitters langgraph pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.5/124.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.0/567.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.5.5
    Uninstalling langchain-core-1.5.5:
      Successfully uninstalled langchain-core-1.5.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are insta

In [5]:
import getpass
import os

openai_api_key = getpass.getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

Enter your OpenAI API Key: ··········


`gpt-3.5-turbo` is used as our LLM in this lab

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Collecting movie introduction data from Wikipedia

In [7]:
# https://python.langchain.com/docs/integrations/document_loaders/wikipedia
# Module Wikipedia is required to use WikipediaLoader
from langchain_community.document_loaders import WikipediaLoader

help(WikipediaLoader)

/tmp/ipykernel_569/2662851850.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


Help on class WikipediaLoader in module langchain_community.document_loaders.wikipedia:

class WikipediaLoader(langchain_core.document_loaders.base.BaseLoader)
 |  WikipediaLoader(query: str, lang: str = 'en', load_max_docs: Optional[int] = 25, load_all_available_meta: Optional[bool] = False, doc_content_chars_max: Optional[int] = 4000)
 |
 |  Load from `Wikipedia`.
 |
 |  The hard limit on the length of the query is 300 for now.
 |
 |  Each wiki page represents one Document.
 |
 |  Method resolution order:
 |      WikipediaLoader
 |      langchain_core.document_loaders.base.BaseLoader
 |      abc.ABC
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, query: str, lang: str = 'en', load_max_docs: Optional[int] = 25, load_all_available_meta: Optional[bool] = False, doc_content_chars_max: Optional[int] = 4000)
 |      Initializes a new instance of the WikipediaLoader class.
 |
 |      Args:
 |          query (str): The query string to search on Wikipedia.
 |       

In [8]:
!pip install wikipedia

  Using cached wikipedia-1.4.0.tar.gz (27 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=f205cb2fac04f58d7fe8bf1fd9da89e022bce031ab0bc232b9d6a86799c36a61
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [10]:
import wikipedia

wikipedia.set_user_agent(
    "RAGBootcamp/1.0 (https://github.com/langchain-ai/langchain)"
)
# load 200 documents from wikipedia
endgame_wikipedia_docs = WikipediaLoader(
    query="Avengers Endgame",
    lang="en",
    load_max_docs=60,  # limit number of downloaded documents. The maximum is 300
    doc_content_chars_max=10000000,
).load()

In [11]:
len(endgame_wikipedia_docs)

60

In [12]:
endgame_wikipedia_docs[0]

Document(metadata={'title': 'Avengers: Endgame', 'summary': "Avengers: Endgame is a 2019 American superhero film based on the Marvel Comics superhero team the Avengers. Produced by Marvel Studios and distributed by Walt Disney Studios Motion Pictures, it is the direct sequel to Avengers: Infinity War (2018) and the 22nd film in the Marvel Cinematic Universe (MCU). Directed by Anthony and Joe Russo and written by Christopher Markus and Stephen McFeely, the film features an ensemble cast that includes Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Don Cheadle, Paul Rudd, Brie Larson, Karen Gillan, Danai Gurira, Benedict Wong, Jon Favreau, Bradley Cooper, Gwyneth Paltrow, and Josh Brolin. In the film, the surviving members of the Avengers and their allies attempt to reverse Thanos's actions in Infinity War, which erased half of all life in the universe.\nThe film was announced in October 2014 as Avengers: Infinity War – Part 2, but Marvel

# Loading Movie Reviews from a CSV file

In [13]:
# https://python.langchain.com/docs/integrations/document_loaders/csv
from langchain_community.document_loaders import CSVLoader

help(CSVLoader)

Help on class CSVLoader in module langchain_community.document_loaders.csv_loader:

class CSVLoader(langchain_core.document_loaders.base.BaseLoader)
 |  CSVLoader(file_path: Union[str, pathlib.Path], source_column: Optional[str] = None, metadata_columns: Sequence[str] = (), csv_args: Optional[Dict] = None, encoding: Optional[str] = None, autodetect_encoding: bool = False, *, content_columns: Sequence[str] = ())
 |
 |  Load a `CSV` file into a list of `Document` objects.
 |
 |  Each document represents one row of the CSV file. Every row is converted
 |  into a key/value pair and outputted to a new line in the document's
 |  page_content.
 |
 |  The source for each document loaded from csv is set to the value of the
 |  `file_path` argument for all documents by default.
 |  You can override this by setting the `source_column` argument to the
 |  name of a column in the CSV file.
 |  The source of each document will then be set to the value of the column
 |  with the name specified in `so

In [15]:
endgame_csv_docs = CSVLoader(
    file_path="/content/endgame.csv", source_column="Review_Url"
).load()

# Spliting documents into chunks
This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

help(RecursiveCharacterTextSplitter)

Help on class RecursiveCharacterTextSplitter in module langchain_text_splitters.character:

class RecursiveCharacterTextSplitter(langchain_text_splitters.base.TextSplitter)
 |  RecursiveCharacterTextSplitter(separators: 'list[str] | None' = None, keep_separator: "bool | Literal['start', 'end']" = True, is_separator_regex: 'bool' = False, **kwargs: 'Any') -> 'None'
 |
 |  Splitting text by recursively look at characters.
 |
 |  Recursively tries to split by different characters to find one
 |  that works.
 |
 |  Method resolution order:
 |      RecursiveCharacterTextSplitter
 |      langchain_text_splitters.base.TextSplitter
 |      langchain_core.documents.transformers.BaseDocumentTransformer
 |      abc.ABC
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, separators: 'list[str] | None' = None, keep_separator: "bool | Literal['start', 'end']" = True, is_separator_regex: 'bool' = False, **kwargs: 'Any') -> 'None'
 |      Create a new TextSplitter.
 |
 |  split_

In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=30,
    length_function=len,
    is_separator_regex=False,
    separators=["\n\n", "\n", ".", " ", ""],  # keep paragraphs, then sentences
)

In [18]:
chunked_endgame_wikipedia_docs = text_splitter.split_documents(endgame_wikipedia_docs)
chunked_endgame_csv_docs = text_splitter.split_documents(endgame_csv_docs)
print(
    f"Number of documents in chunked_endgame_wikipedia_docs: {len(chunked_endgame_wikipedia_docs)}"
)
print(
    f"Number of documents in chunked_endgame_csv_docs: {len(chunked_endgame_csv_docs)}"
)

Number of documents in chunked_endgame_wikipedia_docs: 1346
Number of documents in chunked_endgame_csv_docs: 381


# Embedding and Setting Up Storage With FAISS

In [21]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.1 MB/s eta 0:00:00


In [22]:
from langchain_openai import OpenAIEmbeddings

# https://api.python.langchain.com/en/latest/storage/langchain.storage.file_system.LocalFileStore.html#
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings # https://python.langchain.com/docs/modules/data_connection/text_embedding/caching_embeddings


embeddings_model = OpenAIEmbeddings(model="text-embedding-ada-002")

# Instantiate the LocalFileStore with the local path
file_store = LocalFileStore("./cache/")

embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model, file_store, namespace=embeddings_model.model
)

In [23]:
# Create the vector store with documents:
from langchain_community.vectorstores import FAISS

endgame_csv_db = FAISS.from_documents(chunked_endgame_csv_docs, embedder)
endgame_wiki_db = FAISS.from_documents(chunked_endgame_wikipedia_docs, embedder)

In [24]:
# Set up FAISS retriever:
endgame_csv_faiss_retriever = endgame_csv_db.as_retriever(search_kwargs={"k": 1})
endgame_wiki_faiss_retriever = endgame_wiki_db.as_retriever(search_kwargs={"k": 1})

# Multiple retrievers
In this lab, we will use two retrievers and two sources
1. FAISS retriver
2. BM25 Retriever, a popular ranking function used in information retrieval systems to estimate the relevance of documents to a given search query
    https://python.langchain.com/docs/integrations/retrievers/bm25

Then `Ensemble Retriever` will be used to rerank the results of multiple retrevers in order to achieve better performance.
Sometimes you may want to retrieve documents from multiple different sources, or using multiple different algorithms. The ensemble retriever allows you to easily do this.
https://python.langchain.com/docs/modules/data_connection/retrievers/ensemble

In [25]:
!pip install rank_bm25

  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)


In [26]:
# BM25 retriever:
from langchain_community.retrievers import BM25Retriever

endgame_csv_bm25_retriever = BM25Retriever.from_documents(chunked_endgame_csv_docs, k=1)
endgame_wiki_bm25_retriever = BM25Retriever.from_documents(
    chunked_endgame_wikipedia_docs, k=1
)

In [27]:
# Ensemble retriever with BM25 and FAISS
from langchain_classic.retrievers import EnsembleRetriever

endgame_ensemble_retriever = EnsembleRetriever(
    retrievers=[
        endgame_csv_faiss_retriever,
        endgame_csv_bm25_retriever,
        endgame_wiki_faiss_retriever,
        endgame_wiki_bm25_retriever,
    ],
    weights=[0.25, 0.25, 0.25, 0.25],
)

In [28]:
docs = endgame_ensemble_retriever.invoke("who is the actor of iron man")
docs

[Document(id='31aa938b-d63f-4d85-a776-600fb0351e75', metadata={'source': '/review/rw4824873/?ref_=tt_urv', 'row': 96}, page_content=": 96\nReview_Date: 19 October 2019\nAuthor: rohanumpleby\nRating: 9\nReview_Title: 9.1/10\nReview: In these films the desire and there work rate is fantastic, they do so well together and do well as a team. It is fun to watch them that way. It was a bit boring at times though but still very enjoyable to watch. I can't believe Iron-man has died he was so cool and did very well in his own films. Robert Downey Jr as Iron-Man did so well. And watching the fight at the end was so fun and good to watch. It is good to see that Marvel make good progress as they go along, well most of the time anyway. It is good to also see TOM HOLLAND in the film right at the end to because it gives him a chance as well as the others. He did really well as well like he always dose. And it was funny that Hulk got to have a picture with the kids and Ant-man offers and doesn't get t

# Agents
The core idea of agents is to use a language model to choose a sequence of actions to take. In chains, a sequence of actions is hardcoded (in code). In agents, a language model is used as a reasoning engine to determine which actions to take and in which order.

reference: https://python.langchain.com/docs/modules/agents/

In [29]:
# Tavily(https://tavily.com/) can be used as a search engine to get information online. You have to register to get an API key.
tavily_api_key = getpass.getpass("Enter your Tavily API Key: ")
os.environ["TAVILY_API_KEY"] = tavily_api_key

Enter your Tavily API Key: ··········


In [30]:
# https://docs.tavily.com/docs/tavily-api/langchain
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

search = TavilySearchAPIWrapper()
tavily_tool = TavilySearchResults(max_results=1, api_wrapper=search)
tavily_tool.run("Who is the actor of Iron Man")

/tmp/ipykernel_569/637087643.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=1, api_wrapper=search)


[{'title': 'Iron Man (2008 film)',
  'url': 'https://en.wikipedia.org/wiki/Iron_Man_(2008_film)',
  'content': 'Iron Man is a 2008 American superhero film based on the Marvel Comics character Iron Man. Produced by Marvel Studios and distributed by Paramount Pictures, it is the first film in the Marvel Cinematic Universe (MCU). Directed by Jon Favreau and written by the teams of Mark Fergus & Hawk Ostby and Art Marcum & Matt Holloway, the film stars Robert Downey Jr. as Tony Stark / Iron Man "Tony Stark (Marvel Cinematic Universe)") alongside Terrence Howard, Jeff Bridges, Gwyneth Paltrow, Leslie Bibb, and Shaun Toub. In the film, world-famous industrialist and master engineer Stark builds a mechanized suit of armor "Iron Man\'s armor (Marvel Cinematic Universe)") to escape from captivity by a terrorist group "Ten Rings (organization)"), and then uses the suit to become the superhero Iron Man. [...] Favreau planned to cast a newcomer in the title role, as "those movies don\'t require an

In [31]:
# https://python.langchain.com/docs/modules/agents/quick_start
from langchain_core.tools.retriever import create_retriever_tool

endgame_docs_retrieval_tool = create_retriever_tool(
    retriever=endgame_ensemble_retriever,
    name="Offline_Endgame_Docs_Retriever",  # '^[a-zA-Z0-9_-]{1,64}$'
    description="Searching the Wikipedia and IMDB reviews of Endgame",
)

endgame_retriever_tools = [endgame_docs_retrieval_tool, tavily_tool]

In [32]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# Replace create_conversational_retrieval_agent with LangGraph's create_react_agent
endgame_agent_executor = create_react_agent(llm, endgame_retriever_tools)


/tmp/ipykernel_569/1726291271.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  endgame_agent_executor = create_react_agent(llm, endgame_retriever_tools)


In [33]:
# Use .invoke() instead of calling directly
response = endgame_agent_executor.invoke(
    {"messages": [HumanMessage(content="What happened to Captain America in Endgame at the end?")]}
)

print(response["messages"][-1].content)

At the end of Endgame, Captain America has a significant moment where he says the iconic line "Avengers assemble." This moment was highly anticipated by fans and was considered one of the best scenes in Endgame and the entire Marvel Cinematic Universe (MCU). Additionally, Captain America wields Thor's hammer, Mjolnir, which was a powerful and memorable moment in the film. The scene was well-received by fans in theaters, with many cheering, and critics praised it as one of the defining moments of the "Infinity Saga." Captain America's role in the final battle and his use of the iconic line were seen as a culmination of the character's journey throughout the MCU.


In [34]:
# If we want to keep track of these messages automatically, we can use the following code:
# https://python.langchain.com/docs/expression_language/how_to/message_history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

message_history = ChatMessageHistory()
endgame_agent_with_chat_history = RunnableWithMessageHistory(
    runnable=endgame_agent_executor,
    get_session_history=lambda session_id: message_history,
    input_messages_key="messages",
    history_messages_key="chat_history",
)


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [35]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"session_id": "1"}}

response = endgame_agent_with_chat_history.invoke(
    {"messages": [HumanMessage(content="What happened to Captain America in Endgame at the end?")]},
    config=config
)

print(response["messages"][-1].content)

At the end of Endgame, Captain America has a significant moment where he says the iconic line "Avengers assemble." This moment was highly anticipated by fans and was considered one of the best scenes in Endgame and the entire Marvel Cinematic Universe (MCU). Additionally, Captain America wields Thor's hammer, Mjolnir, which was a powerful and memorable moment in the film. The scene was well-received by fans in theaters, with many cheering, and critics praised it as one of the defining moments of the "Infinity Saga." Captain America's role in the final battle and his use of the iconic line and wielding Mjolnir were highlighted as key moments in the film.


In [36]:
from langchain_core.messages import HumanMessage

response = endgame_agent_with_chat_history.invoke(
    {"messages": [HumanMessage(content="Who was his lover?")]},
    config={"configurable": {"session_id": "2"}}
)

print(response["messages"][-1].content)

I'm not sure who you are referring to. Could you please provide more context or specify the person you are asking about?


In [37]:
from langchain_core.messages import HumanMessage

response = endgame_agent_with_chat_history.invoke(
    {"messages": [HumanMessage(content="Who is Elon Musk?")]},
    config={"configurable": {"session_id": "3"}}
)

print(response["messages"][-1].content)

Elon Musk is a businessman and former public official who is the CEO and largest shareholder of Tesla and SpaceX. He has been the wealthiest person in the world since 2025 and became the only trillionaire in terms of US dollars in June 2026. Musk was born on June 28, 1971, in Pretoria, South Africa, and emigrated to Canada in 1989. He received bachelor's degrees from the University of Pennsylvania before moving to California to pursue business ventures.

Musk co-founded Zip2, an e-commerce payment system that merged with Confinity to form PayPal. He also founded SpaceX in 2002, a space technology company known for innovations in reusable rockets and commercial spaceflight. Musk joined Tesla as an early investor in 2004 and became its CEO and product architect in 2008, leading the company in electric vehicles.

In addition to Tesla and SpaceX, Musk has founded and been involved in various other ventures, including OpenAI, xAI, Twitter (rebranded as X), Neuralink, and The Boring Company.